In [1]:
!pip install datasets

In [7]:
from datasets import load_dataset
!pip install torch
import torch



  Using cached torch-2.10.0-2-cp313-none-macosx_11_0_arm64.whl.metadata (31 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached torch-2.10.0-2-cp313-none-macosx_11_0_arm64.whl (79.5 MB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [torch]32m6/7 [torch]]x]s]


In [13]:
import os
from datasets import load_dataset
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_ID  = "hheiden/PubChem-124M-SMILES-SELFIES-InChI-IUPAC"
OUTPUT_FILE = "dataset/pubchem_data/pubchem_100k.parquet"
COLUMNS     = ["CID", "SMILES_Canonical", "SELFIES", "inchi", "iupac", "formula"]
MAX_SAMPLES = 100_000
BATCH_SIZE  = 256
NUM_WORKERS = 0

# ── Download ──────────────────────────────────────────────────────────────────
def download_100k(output_file=OUTPUT_FILE, columns=COLUMNS):
    if os.path.exists(output_file):
        print(f"Already exists: '{output_file}', skipping download.")
        return
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    print(f"Streaming {MAX_SAMPLES:,} samples from hub ...")
    ds    = load_dataset(DATASET_ID, streaming=True)
    split = ds["train"].take(MAX_SAMPLES)
    records = []
    for i, sample in enumerate(split):
        records.append({c: sample.get(c) for c in columns})
        if (i + 1) % 10_000 == 0:
            print(f"  {i + 1:,} / {MAX_SAMPLES:,}")
    df = pd.DataFrame(records)
    df.to_parquet(output_file, index=False)
    print(f"Saved {len(df):,} rows → '{output_file}' "
          f"({os.path.getsize(output_file) / 1e6:.1f} MB)")

# ── Dataset ───────────────────────────────────────────────────────────────────
class PubChemDataset(Dataset):
    def __init__(self, parquet_file):
        self.df = pd.read_parquet(parquet_file)
        print(f"Loaded {len(self.df):,} rows from '{parquet_file}'")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "cid"    : int(row["CID"])              if pd.notna(row["CID"])              else -1,
            "smiles" : str(row["SMILES_Canonical"]) if pd.notna(row["SMILES_Canonical"]) else "",
            "selfies": str(row["SELFIES"])          if pd.notna(row["SELFIES"])          else "",
            "inchi"  : str(row["inchi"])            if pd.notna(row["inchi"])            else "",
            "iupac"  : str(row["iupac"])            if pd.notna(row["iupac"])            else "",
            "formula": str(row["formula"])          if pd.notna(row["formula"])          else "",
        }

def collate_fn(batch):
    return {
        "cid"    : torch.tensor([b["cid"]     for b in batch], dtype=torch.long),
        "smiles" : [b["smiles"]  for b in batch],
        "selfies": [b["selfies"] for b in batch],
        "inchi"  : [b["inchi"]   for b in batch],
        "iupac"  : [b["iupac"]   for b in batch],
        "formula": [b["formula"] for b in batch],
    }

def make_dataloader(parquet_file, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                    shuffle=True, val_split=0.1):
    dataset = PubChemDataset(parquet_file)
    n       = len(dataset)
    n_val   = int(n * val_split)
    n_train = n - n_val
    train_ds, val_ds = torch.utils.data.random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )
    common = dict(collate_fn=collate_fn, num_workers=num_workers,
                  pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle, **common)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,   **common)
    print(f"Train: {n_train:,} samples ({len(train_loader):,} batches)")
    print(f"Val  : {n_val:,} samples ({len(val_loader):,} batches)")
    return train_loader, val_loader


train_loader, val_loader = make_dataloader(OUTPUT_FILE)

batch = next(iter(train_loader))
print(f"\nBatch keys : {list(batch.keys())}")
print(f"CID tensor : {batch['cid'][:4]}")
print(f"SMILES[0]  : {batch['smiles'][0]}")
print(f"SELFIES[0] : {batch['selfies'][0]}")
print(f"Formula[0] : {batch['formula'][0]}")
print(f"iupac[0] : {batch['iupac'][0]}")

Loaded 100,000 rows from 'dataset/pubchem_data/pubchem_100k.parquet'
Train: 90,000 samples (352 batches)
Val  : 10,000 samples (40 batches)

Batch keys : ['cid', 'smiles', 'selfies', 'inchi', 'iupac', 'formula']
CID tensor : tensor([100014204, 111945606, 158163030, 165710057])
SMILES[0]  : CCc1ccc([C@H]2[C@H](S(C)(=O)=O)[C@@]2(N)C#N)cc1
SELFIES[0] : [C][C][C][=C][C][=C][Branch2][Ring1][Branch1][C@H1][C@H1][Branch1][=Branch2][S][Branch1][C][C][=Branch1][C][=O][=O][C@@][Ring1][#Branch1][Branch1][C][N][C][#N][C][=C][Ring1][S]
Formula[0] : C13H16N2O2S
iupac[0] : (1S,2R,3S)-1-amino-2-(4-ethylphenyl)-3-methylsulfonylcyclopropane-1-carbonitrile


dataset      papers       pubchem_data tp.ipynb
